# NOVAMART — Phase 5 Supply Chain EDA

This notebook uses the approved Phase 3 tables. The public source itself is **synthetic**; suppliers, POs, and receipts are additional **SIMULATED NOVAMART PROCUREMENT SCENARIO** records. It builds descriptive summaries and nine charts. It does not train a model or calculate replenishment policies.

## 1. Reproducible setup and overview
Run all cells from the repository root or this notebook directory. The reusable script writes the charts, CSV summaries, and a JSON metrics file. `units_sold` is fulfilled sales, not verified unconstrained demand.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd()/"data/processed/daily_operations.csv").exists() else Path.cwd().parent
assert (ROOT/"data/processed/daily_operations.csv").exists()
sys.path.insert(0, str(ROOT/"src"))
from phase5_eda import run, load
metrics = run()
d, skus, locations, po, receipts, suppliers = load()
print({k: metrics[k] for k in ["operational_rows", "local_skus", "date_min", "date_max", "total_units_sold", "zero_inventory_days"]})
print("Missing daily values:", int(d.isna().sum().sum()), "Duplicate SKU-days:", int(d.duplicated(["date", "local_sku_id"]).sum()))
display(skus.category.value_counts().rename("local_skus").to_frame())

## 2. Historical sales patterns
The trailing mean smooths day-to-day variation; the monthly chart retains calendar totals. The last month contains only 30 days. Similar weekday means suggest checking whether weekly patterns add predictive value rather than assuming them.

In [ ]:
display(Image(filename=str(ROOT/"images/eda/01_sales_trend.png")))
display(Image(filename=str(ROOT/"images/eda/02_monthly_sales.png")))
display(Image(filename=str(ROOT/"images/eda/03_weekday_sales.png")))

## 3. Category, location, and local-SKU variability
Totals reflect the number of local SKUs in each category as well as sales per SKU. Coefficient of variation is daily standard deviation divided by mean daily units; it can inform later forecast diagnostics and safety-stock assumptions, without calculating those policies here.

In [ ]:
profile = pd.read_csv(ROOT/"reports/sku_demand_profile.csv")
display(profile.sort_values("total_units", ascending=False).head(10))
print("Median CV:", metrics["sku_cv_median"], "Top-10 share:", metrics["top_10_sku_sales_share"])
display(Image(filename=str(ROOT/"images/eda/04_category_location_sales.png")))
display(Image(filename=str(ROOT/"images/eda/05_sku_variability.png")))

## 4. Inventory snapshots and exploratory coverage
Zero inventory is a dated snapshot observation, not a confirmed lost sale. Coverage is latest on-hand divided by the previous 28 calendar days of average sales. Inventory snapshots cannot be reconciled to simulated receipts; this is a screening measure, not a reorder point.

In [ ]:
inventory = pd.read_csv(ROOT/"reports/inventory_eda_summary.csv")
print({k:metrics[k] for k in ["latest_inventory_units","zero_inventory_days","below_7_day_coverage_skus","coverage_median_days"]})
display(inventory.groupby("location_id").agg(latest_units=("inventory_units","sum"),median_coverage_days=("coverage_days_proxy","median"),below_7_days=("coverage_days_proxy",lambda x:int(x.lt(7).sum()))))
display(Image(filename=str(ROOT/"images/eda/06_inventory_distribution_coverage.png")))

## 5. Price, discount, promotion, and correlations
These are observational associations within the published synthetic panel. Promotions were not randomized. Same-day inventory, publisher-estimated demand, weather, and competitor prices may be unavailable at forecast issuance and must not be treated as known future features by default.

In [ ]:
print("Median listed price:",metrics["listed_price_median"])
print("Promotion mean sales:",metrics["promotion_mean_units"])
print("Correlations with sales:",metrics["correlations_with_sales"])
display(Image(filename=str(ROOT/"images/eda/07_promotion_discount_sales.png")))
display(Image(filename=str(ROOT/"images/eda/09_source_correlations.png")))

## 6. SIMULATED NOVAMART PROCUREMENT SCENARIO
The next table and chart use generated POs and receipts, not publisher-observed transactions. Ordered commitment is PO quantity times assumed cost, not payment or realized savings. One receipt exists per generated PO.

In [ ]:
supplier = pd.read_csv(ROOT/"reports/supplier_eda_summary.csv")
display(supplier[["supplier_id","purchase_orders","on_time_pct","late_pct","mean_actual_lead_days","lead_sd_days","receipt_fill_pct","ordered_commitment"]].round(2))
print({k:metrics[k] for k in ["simulated_po_count","simulated_ordered_commitment","simulated_full_receipts","simulated_partial_receipts","simulated_ontime_count","simulated_late_count"]})
display(Image(filename=str(ROOT/"images/eda/08_simulated_supplier_delivery.png")))

## 7. Implications for Phase 6 forecasting
Daily store-local-SKU series are complete, but only about two annual cycles exist. Weekday means are close, while SKU volatility and monthly variation warrant time-aware evaluation. Compare a simple baseline at the provisional 14-day horizon, with 7/30-day sensitivity later. Candidate features include calendar and **past** sales, and only price/promotion values verifiably known at the forecast decision time. Do not use same-day target, inventory, estimated demand, actual weather, or future promotion information without a documented availability rule. This phase creates no forecasting lags or train/test split.

## 8. Implications for Phase 7 inventory optimization
The 406 zero-inventory snapshots and low exploratory coverage merit investigation, but snapshot timing and missing stock movements prevent verified lost-sales or inventory-balance claims. Use measured forecast errors and documented service assumptions later. Supplier lead-time variability is simulated scenario input/outcome, not an empirical operational parameter. No safety stock, EOQ, reorder point, or order recommendation is calculated here.

## 9. Reconciliation
The notebook asserts the expected panel shape. The separate Phase 5 validator independently compares sales, zero stock, and supplier rates to Phase 4 SQL; it also checks chart and summary reproducibility.

In [ ]:
assert len(d)==76000 and d.local_sku_id.nunique()==100 and d.units_sold.sum()==6750876
assert d.inventory_units.eq(0).sum()==406
print("Notebook assertions passed.")